# Add an operation during conversion

First run the [basic PIVview example](01_snapshot_pivview.ipynb). A `udo` is a
callable receiving an open h5rdmtoolbox file after writing and before default
postprocessing and validation. It can be a function; no inheritance is required.
Here we add the magnitude of the two in-plane velocity components, not a 3D speed
or a time average. NaNs propagate from invalid observations.

In [ ]:
from pathlib import Path
import h5py
import numpy as np
from piv2hdf import PIVSnapshot, set_config, tutorial
from piv2hdf.pivview import PIVViewNcFile
from piv2hdf.pivview.user_operations import add_standard_name_operation

def add_inplane_speed(data):
    u, v = data["u"].values[...], data["v"].values[...]
    speed = data.create_dataset(
        "speed", data=np.hypot(u, v),
        attrs={"units": data["u"].attrs["units"],
               "long_name": "Magnitude of in-plane velocity components"},
    )
    speed.dims[0].attach_scale(data["y"])
    speed.dims[1].attach_scale(data["x"])

## Run hooks in order

The built-in hook adds standard names; the second hook adds our field. A snapshot
without `recording_dtime` has no absolute timestamp. Callbacks see a staging file:
do not retain its handle or temporary path. An exception leaves an existing
destination unchanged.

In [ ]:
source = tutorial.PIVview.get_snapshot_nc_files()[0]
reader = PIVViewNcFile(source, udo=[add_standard_name_operation, add_inplane_speed])
with set_config(convention="planar_piv"):
    output = PIVSnapshot(reader).to_hdf(
        hdf_filename=Path("outputs/custom_snapshot.hdf"),
        piv_attributes={"creator": {"name": "Example researcher"}},
    )

In [ ]:
with h5py.File(output, "r") as data:
    np.testing.assert_allclose(data["speed"][...],
                               np.hypot(data["u"][...], data["v"][...]), equal_nan=True)
    print("speed:", data["speed"].shape, data["speed"].attrs["units"])